In [95]:
# ============================================================
# 01 -Silver Processing Parameters
# ============================================================
# Pipeline parameters
# These are fallback values for manual notebook testing.
# When executed from the Master Pipeline, these are overridden
# by the Notebook Activity Base parameters.

batch_file_name = "batch_01_history.csv"
load_run_id = "manual-test"

print(f"Processing batch: {batch_file_name}")
print(f"Load run ID: {load_run_id}")


StatementMeta(, b38494d9-edfe-4e20-9857-ad87cc25a751, 97, Finished, Available, Finished, False)

Processing batch: batch_03_incremental.csv
Load run ID: 18d4edd0-e0d4-452a-840e-652e4a30c291


In [100]:
# ============================================================
# 02- Read the Current Bronze Ingestion
# ============================================================
# Silver processes only the records belonging to the specific
# batch and ingestion run supplied to this notebook.
#
# Filtering by both source_file_name and load_run_id prevents
# previously ingested copies of the same batch from being
# accidentally processed again.

from pyspark.sql import functions as F

bronze_df = (
    spark.table("bronze_sales")
    .filter(
        (F.col("source_file_name") == batch_file_name)
        & (F.col("load_run_id") == load_run_id)
    )
)

bronze_row_count = bronze_df.count()

print(f"Bronze rows selected for processing: {bronze_row_count}")
print(f"Batch: {batch_file_name}")
print(f"Load run ID: {load_run_id}")

if bronze_row_count == 0:
    raise ValueError(
        f"No Bronze records found for batch '{batch_file_name}' "
        f"and load run ID '{load_run_id}'."
    )

StatementMeta(, b38494d9-edfe-4e20-9857-ad87cc25a751, 102, Finished, Available, Finished, False)

Bronze rows selected for processing: 915
Batch: batch_03_incremental.csv
Load run ID: 18d4edd0-e0d4-452a-840e-652e4a30c291


In [101]:
# ============================================================
# 03- Inspect Current Bronze Batch
# ============================================================
# Confirm the structure and sample values received from Bronze
# before applying Silver transformations.

print("=== Bronze Batch Schema ===")
bronze_df.printSchema()

print("\n=== Sample Records ===")
display(bronze_df.limit(5))

StatementMeta(, b38494d9-edfe-4e20-9857-ad87cc25a751, 103, Finished, Available, Finished, False)

=== Bronze Batch Schema ===
root
 |-- row_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- ship_date: string (nullable = true)
 |-- ship_mode: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- sales: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- profit: string (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- load_run_id: string (nullable = true)
 |-- ingestion_timestamp: tim

SynapseWidget(Synapse.DataFrame, 4c79abee-82da-4f5b-ab69-896eb4bbf089)

In [102]:
# ============================================================
# 04- Schema Contract Validation
# ============================================================
# Validate that the CURRENT batch contains the expected
# source columns before it can be promoted further.
#
# Bronze preserves schema evolution across batches. Therefore,
# the Bronze table may contain columns introduced by later
# batches. We validate the columns actually populated in the
# current batch rather than the entire Bronze table schema.
#
# Assessment rule:
# - Missing required columns -> batch failure
# - Unexpected populated columns -> batch failure
# - Column order is not significant
#
# Technical metadata columns created by our pipeline are
# excluded from the source-schema comparison.

expected_source_columns = [
    "row_id",
    "order_id",
    "order_date",
    "ship_date",
    "ship_mode",
    "customer_id",
    "customer_name",
    "segment",
    "country",
    "city",
    "state",
    "postal_code",
    "region",
    "product_id",
    "category",
    "sub_category",
    "product_name",
    "sales",
    "quantity",
    "discount",
    "profit"
]

technical_columns = [
    "source_file_name",
    "load_run_id",
    "ingestion_timestamp"
]

# Determine which source columns are actually populated
# in the CURRENT batch.
#
# This prevents a column introduced by a later batch from
# incorrectly failing an older batch.

actual_source_columns = []

for column in bronze_df.columns:

    if column in technical_columns:
        continue

    # A column is considered received by this batch when
    # at least one record contains a non-null value.
    has_value = (
        bronze_df
        .filter(F.col(column).isNotNull())
        .limit(1)
        .count() > 0
    )

    if has_value:
        actual_source_columns.append(column)

# Identify expected columns that are missing from this batch.
missing_columns = sorted(
    set(expected_source_columns) - set(actual_source_columns)
)

# Identify populated columns that are not part of the
# expected source contract.
unexpected_columns = sorted(
    set(actual_source_columns) - set(expected_source_columns)
)

# Determine the batch-level schema status.
if missing_columns or unexpected_columns:
    schema_status = "FAIL"

    schema_failure_reason = (
        f"Schema validation failed. "
        f"Missing columns: {missing_columns}. "
        f"Unexpected columns: {unexpected_columns}."
    )
else:
    schema_status = "PASS"
    schema_failure_reason = None

print("=== Schema Validation Result ===")
print(f"Batch: {batch_file_name}")
print(f"Status: {schema_status}")
print(f"Populated source columns: {len(actual_source_columns)}")
print(f"Missing columns: {missing_columns}")
print(f"Unexpected columns: {unexpected_columns}")

if schema_failure_reason:
    print(f"Failure reason: {schema_failure_reason}")

StatementMeta(, b38494d9-edfe-4e20-9857-ad87cc25a751, 104, Finished, Available, Finished, False)

=== Schema Validation Result ===
Batch: batch_03_incremental.csv
Status: FAIL
Populated source columns: 22
Missing columns: []
Unexpected columns: ['order_channel']
Failure reason: Schema validation failed. Missing columns: []. Unexpected columns: ['order_channel'].


In [103]:
# ============================================================
# 05- Standardize Silver Data Types
# ============================================================
# Convert source values from their Bronze string representation
# into explicit analytical data types.
#
# The source CSV stores dates in M/d/yyyy format, for example:
# 1/3/2014
#
# Bronze remains unchanged. This creates a new transformed
# DataFrame for Silver processing.

silver_df = (
    bronze_df
    .withColumn("order_date", F.to_date("order_date", "M/d/yyyy"))
    .withColumn("ship_date", F.to_date("ship_date", "M/d/yyyy"))
    .withColumn("sales", F.col("sales").cast("decimal(18,2)"))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("discount", F.col("discount").cast("decimal(18,4)"))
    .withColumn("profit", F.col("profit").cast("decimal(18,2)"))
)

print("Silver data types standardized successfully.")
print("Source date format applied: M/d/yyyy")
silver_df.printSchema()

print("\nSample parsed dates:")
display(
    silver_df
    .select("order_date", "ship_date")
    .where(F.col("order_date").isNotNull())
    .limit(10)
)

StatementMeta(, b38494d9-edfe-4e20-9857-ad87cc25a751, 105, Finished, Available, Finished, False)

Silver data types standardized successfully.
Source date format applied: M/d/yyyy
root
 |-- row_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- ship_date: date (nullable = true)
 |-- ship_mode: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- sales: decimal(18,2) (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount: decimal(18,4) (nullable = true)
 |-- profit: decimal(18,2) (nullable = true)
 |-- source_file_name: string (nullable = true)


SynapseWidget(Synapse.DataFrame, 522e0f34-9f55-458e-a553-50a981e2d2a9)

In [104]:
# ============================================================
# 06- Record-Level Data Quality Validation
# ============================================================
# Identify records that cannot safely be used for reporting.
#
# These are record-level failures, so invalid records are
# isolated rather than automatically failing the entire batch.
#
# Validation rules:
#   1. Row ID must be present.
#   2. Order ID must be present.
#   3. Customer ID must be present.
#   4. Region must be present.
#   5. Sales must be present.
#   6. Sales must be greater than zero.
#
# Each rejected record receives an explicit rejection reason.

dq_df = (
    silver_df
    .withColumn(
        "rejection_reason",
        F.concat_ws(
            "; ",
            F.when(
                F.col("row_id").isNull(),
                F.lit("Missing Row ID")
            ),
            F.when(
                F.col("order_id").isNull(),
                F.lit("Missing Order ID")
            ),
            F.when(
                F.col("customer_id").isNull(),
                F.lit("Missing Customer ID")
            ),
            F.when(
                F.col("region").isNull(),
                F.lit("Missing Region")
            ),
            F.when(
                F.col("sales").isNull(),
                F.lit("Missing Sales")
            ),
            F.when(
                F.col("sales") <= 0,
                F.lit("Invalid Sales: must be greater than zero")
            )
        )
    )
)

# Separate valid and rejected records.
rejected_df = dq_df.filter(
    F.col("rejection_reason") != ""
)

valid_df = dq_df.filter(
    F.col("rejection_reason") == ""
)

# Count the results for validation and observability.
input_row_count = dq_df.count()
rejected_row_count = rejected_df.count()
valid_row_count = valid_df.count()

print("=== Record-Level Data Quality Result ===")
print(f"Input rows: {input_row_count}")
print(f"Valid rows: {valid_row_count}")
print(f"Rejected rows: {rejected_row_count}")

StatementMeta(, b38494d9-edfe-4e20-9857-ad87cc25a751, 106, Finished, Available, Finished, False)

=== Record-Level Data Quality Result ===
Input rows: 915
Valid rows: 795
Rejected rows: 120


In [105]:
# ============================================================
# 7A — Inspect Duplicate Row IDs
# ============================================================
# Before deduplicating the current batch, inspect whether
# duplicate business keys are present.
#
# Row ID is our business key.
# If a Row ID appears more than once within the batch, the
# source has supplied multiple records for the same business
# key. We measure these duplicates before applying the
# deduplication rule in Step 7B.
#
# This is an inspection step only. No data is modified here.
duplicate_row_ids_df = (
    valid_df
    .groupBy("row_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.col("count").desc(), F.col("row_id"))
)

duplicate_row_id_count = duplicate_row_ids_df.count()

print("=== Duplicate Row ID Check ===")
print(f"Duplicate Row IDs found: {duplicate_row_id_count}")

display(duplicate_row_ids_df)

StatementMeta(, b38494d9-edfe-4e20-9857-ad87cc25a751, 107, Finished, Available, Finished, False)

=== Duplicate Row ID Check ===
Duplicate Row IDs found: 15


SynapseWidget(Synapse.DataFrame, ee9e3fbf-bff1-4018-bd11-026956210b70)

In [106]:
# ============================================================
# 7B — Deduplicate Records Within the Current Batch
# ============================================================
# Row ID is the business key for an individual sales record.
#
# If the same Row ID appears more than once within the current
# batch, retain one record so that each business key appears
# only once in the curated Silver input.
#
# The source does not provide a reliable record-version or
# last-updated field, so this step does not claim to identify
# the "latest" duplicate. It simply removes duplicate business
# keys according to the assessment rule.
#
# Corrections arriving in later batches are handled separately
# when the deduplicated batch is MERGED into the Silver
# current-state table using Row ID.
#
# This step only transforms the current batch DataFrame.
# It does not modify the existing Silver current-state table.

before_dedup_count = valid_df.count()

deduped_df = (
    valid_df
    .dropDuplicates(["row_id"])
)

after_dedup_count = deduped_df.count()

duplicate_count = before_dedup_count - after_dedup_count

print("=== Deduplication Result ===")
print(f"Valid rows before deduplication: {before_dedup_count}")
print(f"Rows after deduplication: {after_dedup_count}")
print(f"Duplicate rows removed: {duplicate_count}")

StatementMeta(, b38494d9-edfe-4e20-9857-ad87cc25a751, 108, Finished, Available, Finished, False)

=== Deduplication Result ===
Valid rows before deduplication: 795
Rows after deduplication: 780
Duplicate rows removed: 15


In [107]:
# ============================================================
# 7C — Persist Rejected Records
# ============================================================
# Record-level data-quality failures are not deleted.
# They are isolated into a separate Silver rejection table
# together with the rejection reason and source lineage.
#
# These records are excluded from the reporting/current-state
# Silver table but remain available for investigation and audit.
#
# RERUN BEHAVIOUR:
# If the same batch is processed again, previously stored
# rejected records for that source batch are replaced rather
# than appended again. This keeps the rejection output
# idempotent as well.
#
# IMPORTANT:
# Only records belonging to the current source batch are
# replaced. Rejected records from other batches are preserved.

rejected_output_df = rejected_df

rejected_table_name = "silver_sales_rejected"

if rejected_output_df.count() > 0:

    # If the rejection table does not exist yet, create it.
    if not spark.catalog.tableExists(rejected_table_name):

        (
            rejected_output_df
            .write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(rejected_table_name)
        )

    else:

        # Replace only rejected records belonging to the
        # current source batch.
        (
            rejected_output_df
            .write
            .format("delta")
            .mode("overwrite")
            .option(
                "replaceWhere",
                f"source_file_name = '{batch_file_name}'"
            )
            .option("mergeSchema", "true")
            .saveAsTable(rejected_table_name)
        )

    print("=== Rejected Records Persisted ===")
    print(f"Table: {rejected_table_name}")
    print(f"Rejected rows written: {rejected_output_df.count()}")

else:

    print("=== Rejected Records ===")
    print("No rejected records for this batch.")

StatementMeta(, b38494d9-edfe-4e20-9857-ad87cc25a751, 109, Finished, Available, Finished, False)

=== Rejected Records Persisted ===
Table: silver_sales_rejected
Rejected rows written: 120


In [108]:
# ============================================================
# Verification — Rejection Reason Summary
# ============================================================
# Verify that rejected records are being captured correctly.
#
# A rejected-record table is created only when a batch contains
# rejected records. Therefore, the table may not exist for a
# completely valid batch.
#
# The absence of the table is NOT a pipeline failure.
# It simply means this batch had no rejected records.
#
# The verification is scoped to the current batch and load run,
# so rejected records from earlier batches are not mixed into
# the current verification result.

rejected_table_name = "silver_sales_rejected"

print("=== Rejection Reason Summary ===")

if spark.catalog.tableExists(rejected_table_name):

    rejected_verify_df = (
        spark.table(rejected_table_name)
        .filter(
            (F.col("source_file_name") == batch_file_name)
            & (F.col("load_run_id") == load_run_id)
        )
    )

    display(
        rejected_verify_df
        .groupBy("rejection_reason")
        .count()
        .orderBy(F.col("count").desc())
    )

    print("Rejected-record verification completed.")

else:

    print(
        f"No rejected-record table found. "
        f"This batch produced zero rejected records."
    )

StatementMeta(, b38494d9-edfe-4e20-9857-ad87cc25a751, 110, Finished, Available, Finished, False)

=== Rejection Reason Summary ===


SynapseWidget(Synapse.DataFrame, 9f87dff8-8524-4abc-963b-ca2560393118)

Rejected-record verification completed.


In [109]:
# ============================================================
# 08 — Batch-Level Validation
# ============================================================
# Record-level DQ has already isolated invalid records.
# Here we decide whether the batch as a whole is allowed
# to update the current-state Silver table.
#
# Structural/schema failures stop the entire batch.
# Record-level rejected rows do not necessarily stop the batch.
#
# A failed batch must not modify current-state Silver.
if schema_status == "FAIL":
    batch_status = "FAIL"
    batch_failure_reason = schema_failure_reason

else:
    batch_status = "PASS"
    batch_failure_reason = None

print("=== Batch Validation Result ===")
print(f"Batch: {batch_file_name}")
print(f"Status: {batch_status}")
print(f"Valid rows before deduplication: {valid_row_count}")
print(f"Rejected rows: {rejected_row_count}")
print(f"Duplicate rows removed: {duplicate_count}")
print(f"Rows after deduplication: {after_dedup_count}")

if batch_failure_reason:
    print(f"Failure reason: {batch_failure_reason}")

StatementMeta(, b38494d9-edfe-4e20-9857-ad87cc25a751, 111, Finished, Available, Finished, False)

=== Batch Validation Result ===
Batch: batch_03_incremental.csv
Status: FAIL
Valid rows before deduplication: 795
Rejected rows: 120
Duplicate rows removed: 15
Rows after deduplication: 780
Failure reason: Schema validation failed. Missing columns: []. Unexpected columns: ['order_channel'].


In [110]:
# ============================================================
# 09a — Prepare Current-State Silver Records
# ============================================================
# The current-state Silver table contains only validated,
# deduplicated records.
#
# rejection_reason is removed because rejected records do not
# belong in the current reporting state.
#
# The original source filename, load run ID, and ingestion
# timestamp are retained for lineage and auditability.

silver_current_df = deduped_df.drop("rejection_reason")

print("=== Current-State Silver Dataset ===")
print(f"Rows ready for Silver current state: {silver_current_df.count()}")

display(silver_current_df.limit(5))

StatementMeta(, b38494d9-edfe-4e20-9857-ad87cc25a751, 112, Finished, Available, Finished, False)

=== Current-State Silver Dataset ===
Rows ready for Silver current state: 780


SynapseWidget(Synapse.DataFrame, f4b13e6d-ba44-4954-90b3-f576335d0d84)

In [111]:
# ============================================================
# 9B — Check Silver Current-State Table
# ============================================================

silver_table_name = "silver_sales_current"

table_exists = spark.catalog.tableExists(silver_table_name)

print(f"Silver table: {silver_table_name}")
print(f"Table already exists: {table_exists}")

StatementMeta(, b38494d9-edfe-4e20-9857-ad87cc25a751, 113, Finished, Available, Finished, False)

Silver table: silver_sales_current
Table already exists: True


In [112]:
# ============================================================
# 10 — Verify Batch Metadata
# ============================================================
# Verify that the current batch has the metadata required
# for lineage and auditability.
#
# The batch filename and load run ID are supplied dynamically
# by the pipeline parameters rather than hardcoded.

print("=== Batch Metadata ===")
print(f"Source file: {batch_file_name}")
print(f"Load run ID: {load_run_id}")
print(f"Rows after deduplication: {after_dedup_count}")

StatementMeta(, b38494d9-edfe-4e20-9857-ad87cc25a751, 114, Finished, Available, Finished, False)

=== Batch Metadata ===
Source file: batch_03_incremental.csv
Load run ID: 18d4edd0-e0d4-452a-840e-652e4a30c291
Rows after deduplication: 780


In [113]:
# ============================================================
# Step 10A — Initialize Silver Processing Log
# ============================================================
# This table records the outcome of every Silver processing
# attempt, including successful and failed batches.
#
# It is separate from silver_sales_rejected:
#   - silver_sales_rejected stores individual bad records.
#   - silver_processing_log stores the batch/run outcome.
#
# Keeping this information separately ensures that a failed
# batch still leaves an auditable record even when no data is
# promoted to the current-state Silver table.

silver_log_table_name = "silver_processing_log"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_log_table_name} (
    run_id STRING,
    batch_file_name STRING,
    processing_start_time TIMESTAMP,
    processing_end_time TIMESTAMP,
    status STRING,
    rows_received BIGINT,
    rows_passed BIGINT,
    rows_rejected BIGINT,
    rows_after_deduplication BIGINT,
    rows_written BIGINT,
    failure_reason STRING
)
USING DELTA
""")

print(f"Silver processing log ready: {silver_log_table_name}")

StatementMeta(, b38494d9-edfe-4e20-9857-ad87cc25a751, 115, Finished, Available, Finished, False)

Silver processing log ready: silver_processing_log


In [114]:
# ============================================================
# Step 11 — Initialize or Merge Current-State Silver
# ============================================================
# The current-state Silver table is cumulative.
#
# A failed batch is never promoted to current-state Silver.
# The processing outcome is captured in silver_processing_log
# so both successful and failed runs remain auditable.
#
# Row ID is the business key:
#   - Matching Row ID     -> update existing record
#   - New Row ID          -> insert new record
#
# This MERGE is idempotent for repeated processing of the same
# batch because the same Row ID updates the existing record
# instead of creating another record.



from datetime import datetime
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    LongType
)

processing_start_time = datetime.now()

# Calculate the number of records received for this batch.
input_row_count = bronze_df.count()

rows_written = 0

try:

    # --------------------------------------------------------
    # Batch validation gate
    # --------------------------------------------------------
    # A failed batch must not modify current-state Silver.

    if batch_status != "PASS":
        raise ValueError(
            f"Batch '{batch_file_name}' failed validation. "
            "Current-state Silver will not be updated."
        )

    # --------------------------------------------------------
    # Initialize or MERGE current-state Silver
    # --------------------------------------------------------

    if not spark.catalog.tableExists(silver_table_name):

        # First successful batch: create the current-state table.
        (
            silver_current_df
            .write
            .format("delta")
            .mode("errorifexists")
            .saveAsTable(silver_table_name)
        )

        rows_written = silver_current_df.count()

        print("=== Silver Current-State Initialization ===")
        print(f"Table created: {silver_table_name}")
        print(f"Source batch: {batch_file_name}")
        print(f"Rows written: {rows_written}")

    else:

        # Subsequent successful batches: MERGE using Row ID.
        from delta.tables import DeltaTable

        silver_delta = DeltaTable.forName(
            spark,
            silver_table_name
        )

        (
            silver_delta.alias("target")
            .merge(
                silver_current_df.alias("source"),
                "target.row_id = source.row_id"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

        rows_written = silver_current_df.count()

        print("=== Silver Current-State MERGE ===")
        print(f"Table: {silver_table_name}")
        print(f"Source batch: {batch_file_name}")
        print(f"Rows supplied to MERGE: {rows_written}")
        print("MERGE completed successfully.")

    # --------------------------------------------------------
    # Record successful processing
    # --------------------------------------------------------

    processing_end_time = datetime.now()

    success_log_schema = StructType([
        StructField("run_id", StringType(), True),
        StructField("batch_file_name", StringType(), True),
        StructField("processing_start_time", TimestampType(), True),
        StructField("processing_end_time", TimestampType(), True),
        StructField("status", StringType(), True),
        StructField("rows_received", LongType(), True),
        StructField("rows_passed", LongType(), True),
        StructField("rows_rejected", LongType(), True),
        StructField("rows_after_deduplication", LongType(), True),
        StructField("rows_written", LongType(), True),
        StructField("failure_reason", StringType(), True)
    ])

    success_log_df = spark.createDataFrame(
        [(
            load_run_id,
            batch_file_name,
            processing_start_time,
            processing_end_time,
            "SUCCESS",
            input_row_count,
            valid_row_count,
            rejected_row_count,
            after_dedup_count,
            rows_written,
            None
        )],
        schema=success_log_schema
    )

    (
        success_log_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(silver_log_table_name)
    )

    print("Silver processing log recorded: SUCCESS")

except Exception as e:

    # --------------------------------------------------------
    # Record failed processing
    # --------------------------------------------------------
    # The failure is logged before the exception is re-raised.
    # Re-raising ensures the notebook/pipeline still knows that
    # the batch failed.

    processing_end_time = datetime.now()
    # Prefer the specific batch-validation reason when available.
    # Otherwise, record the actual exception message.
    failure_reason = (
        batch_failure_reason
        if batch_status == "FAIL" and batch_failure_reason
        else str(e)
    )

    failure_log_schema = StructType([
        StructField("run_id", StringType(), True),
        StructField("batch_file_name", StringType(), True),
        StructField("processing_start_time", TimestampType(), True),
        StructField("processing_end_time", TimestampType(), True),
        StructField("status", StringType(), True),
        StructField("rows_received", LongType(), True),
        StructField("rows_passed", LongType(), True),
        StructField("rows_rejected", LongType(), True),
        StructField("rows_after_deduplication", LongType(), True),
        StructField("rows_written", LongType(), True),
        StructField("failure_reason", StringType(), True)
    ])

    failure_log_df = spark.createDataFrame(
        [(
            load_run_id,
            batch_file_name,
            processing_start_time,
            processing_end_time,
            "FAILED",
            input_row_count,
            valid_row_count,
            rejected_row_count,
            after_dedup_count,
            rows_written,
            failure_reason
        )],
        schema=failure_log_schema
    )

    (
        failure_log_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(silver_log_table_name)
    )

    print("Silver processing log recorded: FAILED")
    print(f"Failure reason: {failure_reason}")

    # Important: do not hide the failure.
    raise

StatementMeta(, b38494d9-edfe-4e20-9857-ad87cc25a751, 116, Finished, Available, Finished, False)

Silver processing log recorded: FAILED
Failure reason: Schema validation failed. Missing columns: []. Unexpected columns: ['order_channel'].


ValueError: Batch 'batch_03_incremental.csv' failed validation. Current-state Silver will not be updated.

In [ ]:
# ============================================================
# Step 12 — Verify Current-State Silver
# ============================================================
# Verify that the Silver current-state table exists and that
# Row ID remains unique.
#
# The row count is reported dynamically rather than compared
# with a hardcoded expected value, because batch sizes can vary.

silver_verify_df = spark.table(silver_table_name)

silver_row_count = silver_verify_df.count()

duplicate_silver_row_ids = (
    silver_verify_df
    .groupBy("row_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("=== Current-State Silver Verification ===")
print(f"Silver table: {silver_table_name}")
print(f"Silver row count: {silver_row_count}")
print(f"Duplicate Row IDs: {duplicate_silver_row_ids}")

if duplicate_silver_row_ids != 0:
    raise ValueError(
        f"Silver contains {duplicate_silver_row_ids} duplicate Row IDs."
    )

print("Silver current-state verification PASSED.")

StatementMeta(, b38494d9-edfe-4e20-9857-ad87cc25a751, -1, Cancelled, , Cancelled, True)

In [ ]:
# ============================================================
# Step 13 — Verify Cross-Batch Corrections
# ============================================================
# Compare the current Batch 2 records with the previous Batch 1
# records using Row ID.
#
# A Row ID present in both batches is an existing business
# record. If any business value changed, Batch 2 represents
# a correction to the previous version.
#
# This verification dynamically identifies overlapping and
# corrected Row IDs. The batch filenames are intentionally
# specified because this is an assessment-specific comparison
# of Batch 2 against Batch 1

batch_1_df = (
    spark.table("bronze_sales")
    .filter(
        F.col("source_file_name") == "batch_01_history.csv"
    )
)

batch_2_df = (
    spark.table("bronze_sales")
    .filter(
        F.col("source_file_name") == "batch_02_incremental.csv"
    )
)

# Deduplicate each batch by Row ID for comparison.
batch_1_unique_df = batch_1_df.dropDuplicates(["row_id"])
batch_2_unique_df = batch_2_df.dropDuplicates(["row_id"])

# Identify Row IDs present in both batches.
overlap_df = (
    batch_2_unique_df.alias("b2")
    .join(
        batch_1_unique_df.alias("b1"),
        F.col("b2.row_id") == F.col("b1.row_id"),
        "inner"
    )
)

overlap_count = overlap_df.select("b2.row_id").distinct().count()

# Compare business columns to identify changed records.
comparison_columns = [
    "order_id",
    "order_date",
    "ship_date",
    "ship_mode",
    "customer_id",
    "customer_name",
    "segment",
    "country",
    "city",
    "state",
    "postal_code",
    "region",
    "product_id",
    "category",
    "sub_category",
    "product_name",
    "sales",
    "quantity",
    "discount",
    "profit"
]

change_condition = None

for column_name in comparison_columns:
    condition = ~(
        F.col(f"b2.{column_name}").eqNullSafe(
            F.col(f"b1.{column_name}")
        )
    )

    change_condition = (
        condition
        if change_condition is None
        else change_condition | condition
    )

corrected_df = overlap_df.filter(change_condition)

corrected_count = (
    corrected_df
    .select("b2.row_id")
    .distinct()
    .count()
)

new_record_count = (
    batch_2_unique_df
    .select("row_id")
    .subtract(
        batch_1_unique_df.select("row_id")
    )
    .count()
)

print("=== Cross-Batch Correction Verification ===")
print(f"Batch 2 unique Row IDs: {batch_2_unique_df.select('row_id').distinct().count()}")
print(f"Row IDs present in both batches: {overlap_count}")
print(f"Changed existing Row IDs: {corrected_count}")
print(f"New Row IDs: {new_record_count}")

StatementMeta(, b38494d9-edfe-4e20-9857-ad87cc25a751, -1, Cancelled, , Cancelled, True)

In [ ]:
# ============================================================
# Step 14 — Capture Silver State Before Batch Rerun
# ============================================================
# Capture the current Silver row count before replaying the
# same batch. This gives us a baseline for the idempotency test.

silver_before_rerun_count = (
    spark.table(silver_table_name)
    .count()
)

print("=== Pre-Rerun Silver State ===")
print(f"Silver row count before rerun: {silver_before_rerun_count}")

StatementMeta(, b38494d9-edfe-4e20-9857-ad87cc25a751, -1, Cancelled, , Cancelled, True)

In [ ]:
# ============================================================
# Step 15 — Verify Batch Rerun Idempotency
# ============================================================
# Verify that replaying the same batch did not create
# additional current-state records.
#
# The rerun itself is performed by re-executing the Silver
# processing for the same batch. This cell only validates
# the resulting Silver state.
#
# Row ID remains the business key for the current-state table.

silver_after_rerun_count = (
    spark.table(silver_table_name).count()
)

duplicate_after_rerun = (
    spark.table(silver_table_name)
    .groupBy("row_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("=== Batch Rerun Idempotency Verification ===")
print(f"Silver rows before rerun: {silver_before_rerun_count}")
print(f"Silver rows after rerun: {silver_after_rerun_count}")
print(f"Duplicate Row IDs after rerun: {duplicate_after_rerun}")

if silver_after_rerun_count != silver_before_rerun_count:
    raise ValueError(
        "Idempotency check failed: Silver row count changed "
        "after replaying the same batch."
    )

if duplicate_after_rerun != 0:
    raise ValueError(
        "Idempotency check failed: duplicate Row IDs detected."
    )

print("Silver batch rerun idempotency verification PASSED.")

StatementMeta(, b38494d9-edfe-4e20-9857-ad87cc25a751, -1, Cancelled, , Cancelled, True)